
# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy in moderation. Key points:
    
    - Nutrients: good source of potassium, vitamin C, vitamin B6, and dietary fiber. They also contain resistant starch when unripe.
    - Benefits: support heart health, digestion, and steady energy due to natural sugars and fiber.
    - Considerations: about 90–110 kcal per medium banana; glycemic impact is moderate, can affect blood sugar for some people.
    - Cautions: people with kidney issues or on certain meds (potassium-sparing) should monitor intake; not a complete protein or fat source.
    - Practical: 1–2 bananas daily fit most healthy diets; vary fruit for a range of nutrients.
    
    If you have specific goals or conditions, I can tailor advice.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, nutrient-dense option when eaten in moderation.

Key points:
- Nutrients: good source of potassium, vitamin C, vitamin B6, and fiber.
- Fat/protein: low fat and protein; mainly carbs.
- Typical: ~105 kcal, 27 g carbs, 3 g fiber, 14 g natural sugar, 1 g protein per medium banana.
- Benefits: supports heart health, digestion, and steady energy; contains prebiotic fiber that’s good for gut bacteria.
- Considerations: ripeness affects sugar and resistant starch (green bananas have more resistant starch; as they ripen, sugar increases). Diabetics or kidney concerns should monitor portion size and potassium intake.

Overall: a healthy, convenient snack or part of meals.

_Good Job!_

## Create a Placement Trainer Agent

In [6]:
placement_trainer = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful guide for students preparing for job placements.
    Help with resumes, aptitude and coding practice, technical interviews,
    behavioral and HR interviews, and placement preparation plans.
    Give concise, practical answers using short steps or bullets when useful.
    Tailor your guidance to the student's role, experience, and timeline.
    Ask one focused follow-up question when important context is missing.
    For practice questions, let the student try before revealing the answer
    unless they ask for a solution. Be encouraging, honest, and specific.
    """,
)



In [7]:
with trace("Simple placement Agent"):
    result = await Runner.run(placement_trainer, "How should i prepare for interviews in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Great goal. Here’s a concise 30-day, role-agnostic prep plan you can adapt. It’s structured in weekly blocks with daily action items.
    
    Week 1 — Foundations and target mapping
    - Day 1: Define target roles and companies; list 5 key skills each role requires.
    - Day 2: Update resume (tailor to 2–3 target roles) and create a one-page version.
    - Day 3: Build or refine LinkedIn/GitHub/portfolio (projects with clear outcomes).
    - Day 4: Prepare 3–5 STAR stories for behavioral interviews (teamwork, conflict, failure, leadership).
    - Day 5: Brush up on core technical fundamentals (data structures, algorithms basics relevant to your field).
    - Day 6: Practice 15–20 minutes of coding on platforms (easy–medium) or design notes for system design if applicable.
    - Day 7: Mock interview: 15 minutes with a friend or record yourself answering “Tell me about yourself” and “Why this comp